# Markov Decision Processes (MDPs)

## Learning Objectives
1. Implement a GridWorld MDP from scratch using numpy arrays
2. Define state spaces, action spaces, transition matrices, and reward functions
3. Evaluate a random policy by computing cumulative discounted returns
4. Analyze how discount factor gamma shapes optimal behavior


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

try:
    import torch
    torch.manual_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    device = "cpu"

GAMMA = 0.9
GRID_SIZE = 4
N_STATES = GRID_SIZE * GRID_SIZE
N_ACTIONS = 4  # up, down, left, right
GOAL = 15
HOLES = {5}
print(f"numpy {np.__version__}, torch={TORCH_AVAILABLE}")


## Level 1: Basic GridWorld MDP

In [ ]:
# Basic 4x4 GridWorld MDP: define S, A, P, R matrices
# States: 0..15 (row-major). Goal=15, Hole=5.
# Actions: 0=up, 1=down, 2=left, 3=right

def state_to_rc(s, grid=GRID_SIZE):
    return divmod(s, grid)

def rc_to_state(r, c, grid=GRID_SIZE):
    return r * grid + c

def build_gridworld_mdp(grid_size=4, goal=15, holes=None, slip_prob=0.0):
    # Build transition P[s,a,s'] and reward R[s,a,s'] arrays.
    # P[s,a,s'] = P(s'|s,a), R[s,a,s'] = expected reward for (s,a,s').
    if holes is None:
        holes = set()
    n_s = grid_size * grid_size
    n_a = 4
    P = np.zeros((n_s, n_a, n_s))
    R = np.zeros((n_s, n_a, n_s))
    deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
    for s in range(n_s):
        if s == goal or s in holes:
            P[s, :, s] = 1.0  # Terminal states are absorbing
            continue
        r, c = divmod(s, grid_size)
        for a, (dr, dc) in enumerate(deltas):
            nr = max(0, min(grid_size - 1, r + dr))
            nc = max(0, min(grid_size - 1, c + dc))
            intended = nr * grid_size + nc
            if slip_prob > 0:
                for a2, (dr2, dc2) in enumerate(deltas):
                    nr2 = max(0, min(grid_size - 1, r + dr2))
                    nc2 = max(0, min(grid_size - 1, c + dc2))
                    s2 = nr2 * grid_size + nc2
                    prob = (1 - slip_prob) if a2 == a else slip_prob / 3
                    P[s, a, s2] += prob
            else:
                P[s, a, intended] = 1.0
            for s2 in range(n_s):
                if P[s, a, s2] > 0:
                    if s2 == goal:
                        R[s, a, s2] = 1.0
                    elif s2 in holes:
                        R[s, a, s2] = -1.0
                    else:
                        R[s, a, s2] = -0.01  # Step cost encourages efficiency
    # Verify rows sum to 1
    for s in range(n_s):
        for a_chk in range(n_a):
            total = P[s, a_chk, :].sum()
            assert abs(total - 1.0) < 1e-9, f"P[{s},{a_chk}] sums to {total}"
    return P, R

P, R = build_gridworld_mdp(grid_size=GRID_SIZE, goal=GOAL, holes=HOLES)
print(f"GridWorld MDP built: {N_STATES} states, {N_ACTIONS} actions")
print(f"  P shape: {P.shape},  R shape: {R.shape}")
print(f"  P[0,3,:] (state 0, go right): {P[0,3,:]}")
print(f"  R[14,3,15] (step right to goal): {R[14,3,15]}")


## Level 2: Policy Evaluation -- Iterative V_pi Computation

In [ ]:
# Iterative policy evaluation: V_pi(s) = sum_a pi(a|s)*sum_s' P(s'|s,a)*[R+gamma*V(s')]

def policy_evaluation(pi, P_mat, R_mat, gamma=0.9, theta=1e-4, max_iter=2000):
    # Compute V_pi by repeated Bellman expectation backups until convergence.
    # pi[s,a] = probability of action a in state s.
    # Returns V array and convergence delta history.
    n_s = P_mat.shape[0]
    V = np.zeros(n_s)
    ER = (P_mat * R_mat).sum(axis=2)  # Expected reward E[R|s,a], shape (n_s,n_a)
    deltas = []
    for iteration in range(max_iter):
        # Vectorized Bellman expectation backup
        # For deterministic policy: V_new[s] = ER[s,pi[s]] + gamma * P[s,pi[s],:] @ V
        # For stochastic policy: sum over actions
        future_val = (P_mat * V[None, None, :]).sum(axis=2)  # (n_s, n_a)
        V_new = (pi * (ER + gamma * future_val)).sum(axis=1)  # (n_s,)
        delta = np.max(np.abs(V_new - V))
        deltas.append(delta)
        V = V_new
        if delta < theta:
            print(f"  Converged in {iteration+1} iters (delta={delta:.2e})")
            break
    return V, deltas

# Random policy: uniform over 4 actions
pi_random = np.ones((N_STATES, N_ACTIONS)) / N_ACTIONS
print("Evaluating random policy (gamma=0.9):")
V_random, deltas_r = policy_evaluation(pi_random, P, R, gamma=GAMMA, theta=1e-4)
print(f"  V(start=0) = {V_random[0]:.4f},  V(near_goal=14) = {V_random[14]:.4f}")

# Plot convergence + value heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(deltas_r)
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Max Bellman Residual (log)")
axes[0].set_title("Policy Evaluation Convergence"); axes[0].grid(True, alpha=0.3)
V_grid = V_random.reshape(GRID_SIZE, GRID_SIZE)
im = axes[1].imshow(V_grid, cmap="coolwarm", origin="upper")
for i in range(GRID_SIZE):
    for j in range(GRID_SIZE):
        s = i * GRID_SIZE + j
        label = "G" if s == GOAL else ("H" if s in HOLES else f"{V_random[s]:.2f}")
        axes[1].text(j, i, label, ha="center", va="center", fontsize=8)
axes[1].set_title("V_pi(random) Heatmap"); plt.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.savefig("rl_01_policy_eval.png", dpi=80, bbox_inches="tight")
plt.show(); print("Policy evaluation complete.")


## Real-World Example 1: Inventory Management MDP

In [ ]:
# Inventory MDP: state=stock_level(0-10), action=order_qty(0-5), stochastic demand
import math

def build_inventory_mdp(max_stock=10, max_order=5, demand_mean=3.0):
    # Build inventory management MDP with Poisson demand.
    # State s = current stock. Action a = order quantity.
    # Reward = -(holding_cost*stock + stockout_cost*deficit + order_cost*a)
    n_s = max_stock + 1
    n_a = max_order + 1
    max_demand = max_stock + max_order + 5
    demand_probs = np.array([
        (demand_mean**d) * np.exp(-demand_mean) / math.factorial(d)
        for d in range(max_demand + 1)
    ])
    demand_probs[-1] += max(0.0, 1.0 - demand_probs.sum())
    demand_probs = np.clip(demand_probs, 0, None)
    P_inv = np.zeros((n_s, n_a, n_s))
    R_inv = np.zeros((n_s, n_a, n_s))
    holding_cost, stockout_cost, order_cost = 0.5, 2.0, 1.0
    for s in range(n_s):
        for a in range(n_a):
            after_order = min(s + a, max_stock)
            for d in range(max_demand + 1):
                prob = demand_probs[d]
                if prob < 1e-10:
                    continue
                next_s = max(0, after_order - d)
                stockout = max(0, d - after_order)
                reward = -(holding_cost * next_s + stockout_cost * stockout + order_cost * a)
                P_inv[s, a, next_s] += prob
                R_inv[s, a, next_s] += prob * reward
    mask = P_inv > 1e-10
    R_inv[mask] /= P_inv[mask]
    return P_inv, R_inv

P_inv, R_inv = build_inventory_mdp()
n_inv, n_a_inv = P_inv.shape[0], P_inv.shape[1]
print(f"Inventory MDP: {n_inv} states x {n_a_inv} actions")

def value_iteration(P_vi, R_vi, gamma=0.95, theta=1e-6, max_iter=5000):
    n_s = P_vi.shape[0]
    V = np.zeros(n_s)
    ER_vi = (P_vi * R_vi).sum(axis=2)
    for _ in range(max_iter):
        Q = ER_vi + gamma * (P_vi @ V)
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < theta:
            break
        V = V_new
    return V, (ER_vi + gamma * (P_vi @ V))

V_inv, Q_inv = value_iteration(P_inv, R_inv, gamma=0.95)
pi_inv_opt = np.argmax(Q_inv, axis=1)
print("Optimal order qty by stock level:", pi_inv_opt.tolist())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(n_inv), V_inv, "o-", color="navy")
axes[0].set_xlabel("Stock Level"); axes[0].set_ylabel("V(s)")
axes[0].set_title("Inventory MDP Value Function"); axes[0].grid(True, alpha=0.3)
axes[1].bar(range(n_inv), pi_inv_opt, color="steelblue")
axes[1].set_xlabel("Stock Level"); axes[1].set_ylabel("Order Quantity")
axes[1].set_title("Optimal Policy: Order When Low"); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("rl_01_inventory.png", dpi=80, bbox_inches="tight"); plt.show()


## Real-World Example 2: Routing MDP -- Greedy vs Optimal

In [ ]:
# 5-node routing MDP: stochastic travel times. Compare greedy vs optimal policy.
# State=current_node, action=move_to_node_a, reward=-travel_time, goal=node 4.

def build_routing_mdp(n_nodes=5, gamma=0.95):
    # Edges (from, to, mean_travel_time)
    edges = [(0,1,2),(0,2,5),(1,3,3),(1,2,1),(2,3,2),(2,4,4),(3,4,1)]
    n_a = n_nodes
    P_r = np.zeros((n_nodes, n_a, n_nodes))
    R_r = np.zeros((n_nodes, n_a, n_nodes))
    adj = {i: {} for i in range(n_nodes)}
    for u, v, w in edges:
        adj[u][v] = w
    for s in range(n_nodes):
        if s == n_nodes - 1:
            P_r[s, :, s] = 1.0
            continue
        for a in range(n_a):
            if a in adj[s]:
                mean_t = adj[s][a]
                P_r[s, a, a] = 1.0
                R_r[s, a, a] = -float(mean_t)
            else:
                P_r[s, a, s] = 1.0
                R_r[s, a, s] = -10.0  # Penalty for invalid move
    return P_r, R_r, adj

P_r, R_r, adj_r = build_routing_mdp()
V_r_opt, Q_r_opt = value_iteration(P_r, R_r, gamma=0.95)
pi_r_opt = np.argmax(Q_r_opt, axis=1)

# Greedy policy: pick neighbor with min immediate travel time
def greedy_routing(s, adj_map, n_nodes=5):
    if s == n_nodes - 1 or not adj_map[s]:
        return s
    return min(adj_map[s], key=lambda v: adj_map[s][v])

# Evaluate greedy policy
pi_greedy_r = np.zeros((5, 5))
for s in range(5):
    pi_greedy_r[s, greedy_routing(s, adj_r)] = 1.0
V_r_greedy, _ = policy_evaluation(pi_greedy_r, P_r, R_r, gamma=0.95, theta=1e-6)

print("Routing MDP: V comparison by node (higher = better = less negative cost)")
print(f"  {'Node':>4} {'V_greedy':>12} {'V_optimal':>12} {'Gain':>10}")
for s in range(5):
    gain = V_r_opt[s] - V_r_greedy[s]
    print(f"  {s:>4} {V_r_greedy[s]:>12.3f} {V_r_opt[s]:>12.3f} {gain:>10.3f}")

print(f"\nOptimal routing: {pi_r_opt.tolist()}")
print("  (Greedy ignores downstream path quality; optimal plans full route)")


## Real-World Example 3: Discount Factor Sweep

In [ ]:
# How gamma shapes optimal policy: sweep gamma 0.5 -> 0.99 on GridWorld
gammas = [0.5, 0.7, 0.9, 0.95, 0.99]
results = {}
for g in gammas:
    V_g, Q_g = value_iteration(P, R, gamma=g)
    pi_g = np.argmax(Q_g, axis=1)
    results[g] = {"V": V_g, "pi": pi_g}

action_names = ["Up", "Down", "Left", "Right"]
print(f"{'Gamma':>7} | {'V(start=0)':>12} | {'Action@s0':>12} | {'V(s=14)':>10}")
print("-" * 52)
for g in gammas:
    v0 = results[g]["V"][0]
    a0 = results[g]["pi"][0]
    v14 = results[g]["V"][14]
    print(f"  {g:.2f}  | {v0:>12.4f} | {action_names[a0]:>12} | {v14:>10.4f}")

# Plot V(s) vs gamma for key states
key_states = [0, 3, 12, 14]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for s in key_states:
    axes[0].plot(gammas, [results[g]["V"][s] for g in gammas], "o-", label=f"s={s}", linewidth=1.6)
axes[0].set_xlabel("Discount Factor gamma"); axes[0].set_ylabel("V(s)")
axes[0].set_title("State Values vs Discount Factor"); axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Policy comparison: simulate episodes for random/greedy/optimal policies
def run_episodes(pi_idx, P_sim, R_sim, gamma=0.9, n_ep=200, max_steps=100):
    n_s = P_sim.shape[0]
    returns = []
    for _ in range(n_ep):
        s = 0; G = 0.0; disc = 1.0
        for _ in range(max_steps):
            a = pi_idx[s]
            s_next = np.random.choice(n_s, p=P_sim[s, a, :])
            G += disc * R_sim[s, a, s_next]
            disc *= gamma; s = s_next
            if s == GOAL or s in HOLES:
                break
        returns.append(G)
    return np.mean(returns)

ER_gw = (P * R).sum(axis=2)
pi_star_idx = np.argmax((ER_gw + GAMMA * (P @ results[0.9]["V"])), axis=1)
pi_greedy_gw = np.argmax(ER_gw, axis=1)
pi_rand_gw = np.array([np.random.randint(N_ACTIONS) for _ in range(N_STATES)])

policy_means = {
    "Random": run_episodes(pi_rand_gw, P, R),
    "Greedy (1-step)": run_episodes(pi_greedy_gw, P, R),
    "Optimal": run_episodes(pi_star_idx, P, R),
}
colors = ["#c0392b", "#f39c12", "#27ae60"]
axes[1].bar(list(policy_means.keys()), list(policy_means.values()), color=colors, width=0.5, alpha=0.85)
V_star_s0 = results[0.9]["V"][0]
axes[1].axhline(V_star_s0, color="navy", linestyle="--", label=f"V*(s0)={V_star_s0:.3f}")
axes[1].set_ylabel("Average Discounted Return"); axes[1].set_title("Policy Comparison (gamma=0.9)")
axes[1].legend(); axes[1].grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("rl_01_gamma_sweep.png", dpi=80, bbox_inches="tight"); plt.show()


# --- Policy arrows visualization for gamma=0.9 ---
action_arrows = {0: (0, -0.4), 1: (0, 0.4), 2: (-0.4, 0), 3: (0.4, 0)}  # up/down/left/right
pi_opt_90 = results[0.9]["pi"]

fig_arrows, ax_arr = plt.subplots(figsize=(5, 5))
for s in range(N_STATES):
    r, c_pos = divmod(s, GRID_SIZE)
    if s == GOAL:
        ax_arr.add_patch(plt.Rectangle((c_pos-0.45, r-0.45), 0.9, 0.9, color="gold"))
        ax_arr.text(c_pos, r, "G", ha="center", va="center", fontsize=12, fontweight="bold")
    elif s in HOLES:
        ax_arr.add_patch(plt.Rectangle((c_pos-0.45, r-0.45), 0.9, 0.9, color="#c0392b", alpha=0.7))
        ax_arr.text(c_pos, r, "H", ha="center", va="center", fontsize=12, color="white")
    else:
        a = pi_opt_90[s]
        dx, dy = action_arrows[a]
        ax_arr.annotate("", xy=(c_pos + dx, r - dy),
                        xytext=(c_pos, r),
                        arrowprops=dict(arrowstyle="->", color="navy", lw=2.0))
        ax_arr.text(c_pos + 0.3, r - 0.35, f"{results[0.9]['V'][s]:.2f}",
                    fontsize=6, color="gray")

ax_arr.set_xlim(-0.6, GRID_SIZE - 0.4); ax_arr.set_ylim(-0.6, GRID_SIZE - 0.4)
ax_arr.set_xticks(range(GRID_SIZE)); ax_arr.set_yticks(range(GRID_SIZE))
ax_arr.set_aspect("equal"); ax_arr.invert_yaxis()
ax_arr.set_title("Optimal Policy Arrows (gamma=0.9)\nNumbers = V*(s)")
ax_arr.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("rl_01_arrows.png", dpi=80, bbox_inches="tight"); plt.show()
print("Policy arrows: state 0 -> Right (towards goal); state 5 is a hole (skipped).")

# --- Cumulative reward curves over 100 episodes ---
def run_cumulative_episodes(pi_idx, P_sim, R_sim, gamma=0.9, n_ep=100, max_steps=60):
    # Returns array of per-episode discounted returns
    n_s = P_sim.shape[0]
    ep_returns = []
    for ep in range(n_ep):
        s = 0; G = 0.0; disc = 1.0
        for _ in range(max_steps):
            a = pi_idx[s]
            s_next = int(np.random.choice(n_s, p=P_sim[s, a, :]))
            G += disc * R_sim[s, a, s_next]
            disc *= gamma; s = s_next
            if s == GOAL or s in HOLES:
                break
        ep_returns.append(G)
    return np.array(ep_returns)

np.random.seed(42)
returns_rand = run_cumulative_episodes(pi_rand_gw, P, R, n_ep=100)
returns_greedy = run_cumulative_episodes(pi_greedy_gw, P, R, n_ep=100)
returns_opt = run_cumulative_episodes(pi_star_idx, P, R, n_ep=100)

window = 10  # smoothing window
def smooth(arr, w):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

fig_curves, ax_c = plt.subplots(figsize=(10, 4))
ax_c.plot(smooth(returns_rand, window), color="#c0392b", label="Random policy", alpha=0.85)
ax_c.plot(smooth(returns_greedy, window), color="#f39c12", label="Greedy (1-step)", alpha=0.85)
ax_c.plot(smooth(returns_opt, window), color="#27ae60", label="Optimal policy", alpha=0.85)
ax_c.axhline(results[0.9]["V"][0], color="navy", linestyle="--", lw=1.5, label=f"V*(s0)={results[0.9]['V'][0]:.3f}")
ax_c.set_xlabel("Episode"); ax_c.set_ylabel("Discounted Return G (smoothed 10-ep)")
ax_c.set_title("Cumulative Return Curves: Random vs Greedy vs Optimal (gamma=0.9)")
ax_c.legend(fontsize=9); ax_c.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("rl_01_curves.png", dpi=80, bbox_inches="tight"); plt.show()
print(f"Mean returns -- Random: {returns_rand.mean():.3f}  Greedy: {returns_greedy.mean():.3f}  Optimal: {returns_opt.mean():.3f}")


## Comparison: Policy Values Across Strategies and Gamma

In [ ]:
# Side-by-side comparison: gamma sweep results + policy quality bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# V(s=0) vs gamma for all three policy types
# (Re-run episodes for each gamma to show how policy quality changes)
gamma_sweep_results = {}
for g in gammas:
    V_g_check, Q_g_check = value_iteration(P, R, gamma=g)
    pi_opt_g = np.argmax(Q_g_check, axis=1)
    pi_greedy_g = np.argmax((P * R).sum(axis=2), axis=1)
    pi_rand_g = np.array([np.random.randint(N_ACTIONS) for _ in range(N_STATES)])
    gamma_sweep_results[g] = {
        "V_opt": V_g_check[0],
        "emp_opt": run_episodes(pi_opt_g, P, R, gamma=g, n_ep=150),
        "emp_greedy": run_episodes(pi_greedy_g, P, R, gamma=g, n_ep=150),
        "emp_rand": run_episodes(pi_rand_g, P, R, gamma=g, n_ep=150),
    }

axes[0].plot(gammas, [gamma_sweep_results[g]["V_opt"] for g in gammas],
             "o-", color="navy", label="V*(s0) from DP", linewidth=2)
axes[0].plot(gammas, [gamma_sweep_results[g]["emp_opt"] for g in gammas],
             "s--", color="green", label="Empirical (optimal pi)", linewidth=1.5)
axes[0].plot(gammas, [gamma_sweep_results[g]["emp_greedy"] for g in gammas],
             "^:", color="orange", label="Empirical (greedy pi)", linewidth=1.5)
axes[0].set_xlabel("Discount Factor gamma"); axes[0].set_ylabel("Value V(start=0)")
axes[0].set_title("V*(s0) vs Gamma: DP vs Simulation"); axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Summary table of final values
print("Discount factor impact on GridWorld (start state s=0):")
print(f"  {'Gamma':>6} {'DP V*(s0)':>12} {'Opt(sim)':>10} {'Greedy(sim)':>12} {'Random(sim)':>12}")
for g in gammas:
    r = gamma_sweep_results[g]
    print(f"  {g:>6.2f} {r['V_opt']:>12.4f} {r['emp_opt']:>10.4f} "
          f"{r['emp_greedy']:>12.4f} {r['emp_rand']:>12.4f}")

# Heatmap: V*(s) at gamma=0.99
V_99, Q_99 = value_iteration(P, R, gamma=0.99)
im = axes[1].imshow(V_99.reshape(GRID_SIZE, GRID_SIZE), cmap="viridis", origin="upper")
for i in range(GRID_SIZE):
    for j in range(GRID_SIZE):
        s = i * GRID_SIZE + j
        label = "G" if s == GOAL else ("H" if s in HOLES else f"{V_99[s]:.2f}")
        axes[1].text(j, i, label, ha="center", va="center", fontsize=7, color="white")
axes[1].set_title("V*(s) at gamma=0.99"); plt.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.savefig("rl_01_comparison.png", dpi=80, bbox_inches="tight"); plt.show()


# --- Value function 4x4 heatmap for gamma=0.9 ---
V_90 = results[0.9]["V"]
fig_hm, axes_hm = plt.subplots(1, 2, figsize=(12, 5))

im_90 = axes_hm[0].imshow(V_90.reshape(GRID_SIZE, GRID_SIZE), cmap="Blues", origin="upper")
for i in range(GRID_SIZE):
    for j in range(GRID_SIZE):
        s = i * GRID_SIZE + j
        label = "G" if s == GOAL else ("H" if s in HOLES else f"{V_90[s]:.3f}")
        color = "black" if V_90[s] < 0.5 * V_90.max() else "white"
        axes_hm[0].text(j, i, label, ha="center", va="center", fontsize=9, color=color)
axes_hm[0].set_title("V*(s) Heatmap — gamma=0.9"); plt.colorbar(im_90, ax=axes_hm[0])
axes_hm[0].set_xticks(range(GRID_SIZE)); axes_hm[0].set_yticks(range(GRID_SIZE))

# Compare V*(s=0) across three gamma values with error annotation
selected_gammas = [0.5, 0.9, 0.99]
v0_vals = [results[g]["V"][0] for g in selected_gammas]
bar_colors = ["#3498db", "#2ecc71", "#e74c3c"]
bars = axes_hm[1].bar([str(g) for g in selected_gammas], v0_vals, color=bar_colors, alpha=0.8, width=0.5)
for bar, val in zip(bars, v0_vals):
    axes_hm[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
axes_hm[1].set_xlabel("Discount Factor gamma"); axes_hm[1].set_ylabel("V*(s=0)")
axes_hm[1].set_title("Start-State Value vs Discount Factor")
axes_hm[1].grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("rl_01_heatmap_compare.png", dpi=80, bbox_inches="tight"); plt.show()

# Effective planning horizon 1/(1-gamma)
print("\nEffective planning horizons:")
for g in [0.5, 0.7, 0.9, 0.95, 0.99]:
    horizon = 1.0 / (1.0 - g)
    print(f"  gamma={g:.2f}: horizon = {horizon:.1f} steps")
print("\nHigher gamma -> agent values future rewards -> longer planned paths -> higher V*(s).")


## Key Takeaways

**Core idea:** An MDP (S, A, P, R, gamma) formalizes sequential decision-making. The Markov
property means next state depends only on current state and action, enabling exact DP solutions.
The discount factor gamma balances myopic vs long-horizon reward.

**Discount factor effects:**
| gamma | Horizon | Behavior |
|-------|---------|----------|
| 0.5 | Short | Prefers immediate rewards; avoids long paths |
| 0.9 | Medium | Standard episodic tasks |
| 0.99 | Long | Plans far ahead; V(s) increases significantly |

**Common failure modes:**
- Terminal state V not kept at 0 -> inflated values propagate
- Transition matrix rows not summing to 1 -> wrong policy
- gamma=1 in continuing tasks -> V diverges

**Related concepts:**
- [Bellman Equations](./02-bellman-equations.ipynb) - recursive value computation
- [Dynamic Programming for RL](./03-dynamic-programming-rl.ipynb) - algorithms using known P, R
- [Monte Carlo Methods](./04-monte-carlo-methods.ipynb) - model-free alternative
- [TD Learning](./05-temporal-difference-learning.ipynb) - bootstrapped model-free learning


## Exercises

1. **Stochastic transitions**: Rebuild GridWorld with `slip_prob=0.2`. Compare V*(s=0) to
   the deterministic version. Which states change most?
2. **Reward shaping**: Change step penalty from -0.01 to -0.5. Does the optimal policy change?
3. **Larger grid**: Scale to 8x8 (64 states, 4 actions). How does V*(s=0) change?
4. **Gamma sensitivity**: Find the minimum gamma at which the policy prefers going around the
   hole vs taking a direct but risky path through state 5.
